# Pooled Bayesian estimates

Source notebook for table 2

PTCy meta-analysis collaboration

Table 2 of the article reports the posterior median odds ratio, its 95% credible interval, and the between-study standard deviation τ for every outcome and comparison. Those posteriors come from `brms` fits that take hours to run and are not re-fitted here. What this notebook does is take the **exported posterior summaries** and format them into the published table, so that the table and the model output cannot drift apart.

Inputs, from `03_models/set_b/` of the analysis repository:

- `data/models/Table2_setB.csv` — posterior summaries for the adopted (Set B, cohort-deduplicated) analysis
- `data/models/freq_results_setB.csv` — frequentist REML concordance checks

In [ ]:
library(dplyr)



Attaching package: 'dplyr'

The following objects are masked from 'package:stats':

    filter, lag

The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union

Rows: 29
Columns: 9
$ slug      <chr> "c1_agvhd", "c1_bk", "c1_bsi", "c1_cgvhd_any", "c1_cgvhd_ms"…
$ model     <chr> "m1", "m1", "m1", "m1", "m1", "m1", "m2_steroid", "m1", "m1"…
$ k         <dbl> 28, 11, 6, 8, 19, 22, 22, 6, 13, 12, 12, 35, 28, 34, 27, 8, …
$ n_total   <dbl> 4197, 1333, 806, 1732, 2840, 3328, 3328, 2243, 5504, 1309, 1…
$ or_median <dbl> 0.6749797, 2.4756792, 1.8699270, 0.2329979, 0.4651549, 1.257…
$ ci_low    <dbl> 0.5865728, 1.8229454, 1.3269970, 0.1814588, 0.3803600, 1.074…
$ ci_high   <dbl> 0.7784357, 3.3845067, 2.6232976, 0.2971835, 0.5667511, 1.469…
$ tau       <dbl> 0.6670596, 0.9990599, 0.8769372, 0.5164964, 0.7657332, 0.728…
$ source    <chr> "unchanged", "unchanged", "unchanged", "unchanged", "Set B (…

## Formatting the published table

Odds ratios are set to two decimal places with the middle dot the journal uses for decimals, and totals are set with a thin space as the thousands separator.

In [ ]:

middot  <- function(x, d = 2) str_replace(sprintf("%.*f", d, x), fixed("."), "·")
or_cri  <- function(or, lo, hi) sprintf("%s (%s–%s)", middot(or), middot(lo), middot(hi))
thin_n  <- function(n) format(n, big.mark = " ", trim = TRUE)

# slug -> (outcome label, comparison), in the order the article presents them
key <- tribble(
  ~slug,          ~outcome,                      ~comparison,
  "c1_os",        "Overall survival",            "C1",
  "c2_os",        "Overall survival",            "C2",
  "c3_os",        "Overall survival",            "C3",
  "c1_nrm",       "Non-relapse mortality",       "C1",
  "c3_nrm",       "Non-relapse mortality",       "C3",
  "c1_rrm",       "Relapse-related mortality",   "C1",
  "c2_rrm",       "Relapse-related mortality",   "C2",
  "c3_rrm",       "Relapse-related mortality",   "C3",
  "c1_agvhd",     "Acute GVHD II–IV",       "C1",
  "c2_agvhd",     "Acute GVHD II–IV",       "C2",
  "c3_agvhd",     "Acute GVHD II–IV",       "C3",
  "c1_cgvhd_ms",  "Chronic GVHD mod–severe","C1",
  "c2_cgvhd_ms",  "Chronic GVHD mod–severe","C2",
  "c1_cmv",       "CMV reactivation",            "C1",
  "c2_cmv",       "CMV reactivation",            "C2",
  "c1_bsi",       "Bloodstream infection",       "C1",
  "c1_ifi_any",   "Invasive fungal infection",   "C1",
  "c1_bk",        "BK virus",                    "C1",
  "c2_bk",        "BK virus",                    "C2",
  "c1_irm",       "Infection-related mortality", "C1",
  "c2_irm",       "Infection-related mortality", "C2")

# GRADE ratings are a judgement recorded during the review, not a model output
grade <- tribble(
  ~slug,         ~model, ~GRADE,
  "c1_os",       "m1",   "Low",
  "c2_os",       "m1",   "Low",
  "c1_nrm",      "m1",   "Very low",
  "c1_agvhd",    "m1",   "Low",
  "c2_agvhd",    "m1",   "Low",
  "c1_cmv",      "m1",   "Low",
  "c2_cmv",      "m1",   "Very low",
  "c1_bsi",      "m1",   "Very low",
  "c1_ifi_any",  "m1",   "Very low")


In [ ]:

t2 |>
  inner_join(key, by = "slug") |>
  left_join(grade, by = c("slug", "model")) |>
  mutate(
    Model = recode(model, m1 = "M1", m2_steroid = "M2"),
    order_out = match(outcome, unique(key$outcome))) |>
  arrange(order_out, comparison, Model) |>
  transmute(
    Outcome    = outcome,
    Comparison = comparison,
    Model,
    k,
    N          = thin_n(n_total),
    `OR (95% CrI)` = or_cri(or_median, ci_low, ci_high),
    `τ`   = middot(tau),
    GRADE      = coalesce(GRADE, "—"),
    Source     = source) |>
  kable(align = "llcrrlrll")


  ----------------------------------------------------------------------------------------------------------
  Outcome             Comparison    Model     k        N OR (95% CrI)        τ GRADE   Source
  ------------------- ------------ ------- ---- -------- -------------- ------ ------- ---------------------
  Overall survival    C1             M1      35   10 133 0·84             0·66 Low     Set B (deduplicated)
                                                         (0·76--0·92)                  

  Overall survival    C1             M2      28   10 133 1·04             0·60 ---     Set B (deduplicated)
                                                         (0·89--1·22)                  

  Overall survival    C2             M1       9   12 791 0·83             0·50 Low     Set B (deduplicated)
                                                         (0·75--0·92)                  

  Overall survival    C2             M2       8   12 791 0·81             0·67 ---     Set B (deduplicated)
                                                         (0·70--0·93)                  

  Overall survival    C3             M1       6    4 746 1·11             0·84 ---     unchanged
                                                         (0·95--1·30)                  

  Non-relapse         C1             M1      12    1 309 0·88             0·69 Very    unchanged
  mortality                                              (0·66--1·18)          low     

  Non-relapse         C1             M2      12    1 309 0·90             0·56 ---     unchanged
  mortality                                              (0·59--1·36)                  

  Non-relapse         C3             M1       5    2 529 0·96             1·11 ---     unchanged
  mortality                                              (0·74--1·24)                  

  Relapse-related     C1             M1      34   11 121 0·86             0·47 ---     Set B (deduplicated)
  mortality                                              (0·77--0·97)                  

  Relapse-related     C1             M2      27   11 121 0·85             0·55 ---     Set B (deduplicated)
  mortality                                              (0·73--0·99)                  

  Relapse-related     C2             M1      10   17 182 0·79             0·14 ---     unchanged
  mortality                                              (0·69--0·89)                  

  Relapse-related     C3             M1       6      692 0·80             0·31 ---     unchanged
  mortality                                              (0·50--1·27)                  

  Acute GVHD II--IV   C1             M1      28    4 197 0·67             0·67 Low     unchanged
                                                         (0·59--0·78)                  

  Acute GVHD II--IV   C2             M1       8    1 039 0·63             0·53 Low     Set B (deduplicated)
                                                         (0·47--0·83)                  

  Acute GVHD II--IV   C3             M1       7    2 450 1·00             0·75 ---     unchanged
                                                         (0·80--1·25)                  

  Chronic GVHD        C1             M1      19    2 840 0·47             0·77 ---     Set B (deduplicated,
  mod--severe                                            (0·38--0·57)                  spec corrected)

  Chronic GVHD        C2             M1       6      905 0·79             0·89 ---     unchanged
  mod--severe                                            (0·55--1·14)                  

  CMV reactivation    C1             M1      22    3 328 1·26             0·73 Low     unchanged
                                                         (1·07--1·47)                  

  CMV reactivation    C1             M2      22    3 328 1·25             0·74 ---     unchanged
                                                         (1·01--1·54)                  

  CMV reactivation    C2             M1      12    1 511 0·97             0·63 Very    Set B (deduplicated)
                                                         (0·77--1·23)          low     

  CMV reactivation    C2             M2      12    1 511 1·00             0·72 ---     Set B (deduplicated)
                                                         (0·79--1·26)                  

  Bloodstream         C1             M1       6      806 1·87             0·88 Very    unchanged
  infection                                              (1·33--2·62)          low     

  Invasive fungal     C1             M1       6    2 243 0·43             1·53 Very    unchanged
  infection                                              (0·29--0·63)          low     

  BK virus            C1             M1      11    1 333 2·48             1·00 ---     unchanged
                                                         (1·82--3·38)                  

  BK virus            C2             M1       3      313 2·44             0·39 ---     unchanged
                                                         (1·40--4·26)                  

  Infection-related   C1             M1      13    5 504 1·19             0·82 ---     Set B (deduplicated)
  mortality                                              (1·00--1·43)                  

  Infection-related   C2             M1       7   13 346 0·90             0·58 ---     unchanged
  mortality                                              (0·74--1·09)                  
  ----------------------------------------------------------------------------------------------------------


## Frequentist concordance check

Every Bayesian estimate was re-fitted with restricted maximum likelihood (`metafor`) as a concordance check. Agreement in direction and rough magnitude — not identical point estimates — is what this check is for; the Bayesian intervals are wider because they propagate uncertainty in τ.

In [ ]:

fcols <- names(freq)
cat("freq_results_setB.csv columns:", paste(fcols, collapse = ", "), "\n\n")


freq_results_setB.csv columns: slug, source, k, freq_or, freq_lo, freq_hi, I2, tau_freq, Q_p 

  ----------------------------------------------------------------------------------------------------------
  slug           source           k     freq_or     freq_lo     freq_hi           I2    tau_freq         Q_p
  -------------- ------------- ---- ----------- ----------- ----------- ------------ ----------- -----------
  c1_agvhd       post_block9     28   0.6085577   0.4664612   0.7939407   62.0684173   0.5256999   0.0000519

  c1_bk          post_block9     11   2.3613993   1.6303462   3.4202592   17.4328194   0.2567734   0.2293894

  c1_bsi         post_block9      6   1.2920808   0.6118992   2.7283462   71.3231492   0.7535551   0.0099051

  c1_cgvhd_any   post_block9      8   0.2449557   0.1506285   0.3983527   62.8579128   0.5134226   0.0007805

  c1_cgvhd_ms    set_b           19   0.4163445   0.2846562   0.6089547   57.7336187   0.5761175   0.0006579

  c1_cmv         post_block9     22   1.2489584   0.9383597   1.6623657   61.9243742   0.5028618   0.0000207

  c1_ifi_any     post_block9      6   0.6048511   0.2101370   1.7409823   80.4854135   1.1051332   0.0005379

  c1_irm         set_b           13   1.0253110   0.7181718   1.4638039   57.2420420   0.4478379   0.0067851

  c1_nrm         post_block9     12   0.9138251   0.6787301   1.2303511    0.0004339   0.0011633   0.4275747

  c1_os          set_b           35   0.7859317   0.6542028   0.9441852   64.0273006   0.3953783   0.0000127

  c1_rrm         set_b           34   0.8595756   0.7618077   0.9698908    0.0000000   0.0000000   0.4783104

  c2_agvhd       set_b            8   0.5103917   0.2757743   0.9446122   73.9074663   0.7337996   0.0000509

  c2_bk          post_block9      3   2.4438528   1.4146224   4.2219155    0.0000000   0.0000000   0.7723960

  c2_cgvhd_ms    post_block9      6   0.7665687   0.4044426   1.4529318   59.6798518   0.5741171   0.0281422

  c2_cmv         set_b           12   0.8279625   0.5141315   1.3333592   69.2478069   0.6613407   0.0002630

  c2_irm         post_block9      7   0.9319776   0.6646383   1.3068495   38.5949926   0.2517628   0.1052105

  c2_os          set_b            9   0.7326234   0.4429834   1.2116413   92.9510966   0.6735393   0.0000000

  c2_rrm         post_block9     10   0.7752236   0.5737345   1.0474735   57.0224978   0.2959984   0.0179973

  c3_agvhd       post_block9      7   0.5192072   0.2246361   1.2000568   77.3153276   0.8761138   0.0000008

  c3_cgvhd       post_block9      4   0.4965658   0.1889736   1.3048254   48.8113187   0.6836530   0.1190161

  c3_nrm         post_block9      5   0.4919976   0.1739930   1.3912152   80.0066700   0.9519983   0.0000046

  c3_os          post_block9      6   0.9199758   0.5681119   1.4897688   78.1079841   0.4490891   0.0012072

  c3_rrm         post_block9      6   0.7524179   0.4386457   1.2906375    6.9717560   0.2000115   0.5579643
  ----------------------------------------------------------------------------------------------------------


## Publication bias

In [ ]:

f <- here("data", "models", "pub_bias_results.csv")
if (file.exists(f)) read_csv(f, show_col_types = FALSE) |> kable() else
  cat("pub_bias_results.csv not vendored into this project.\n")


  ---------------------------------------------------------------------------------------------------------------------
  Outcome               k   raw_OR   egger_std_p    peters_p      begg_p   tf_k_imp   tf_adj_or   tf_adj_lo   tf_adj_hi
  ------------------ ---- -------- ------------- ----------- ----------- ---------- ----------- ----------- -----------
  Overall survival     35     0.75     0.2985854   0.9946418   0.1396095          0        0.75        0.63        0.89

  Non-relapse          15     0.78     0.0359400   0.3605873   0.0926426          3        0.89        0.62        1.28
  mortality                                                                                                 

  aGVHD grade II--IV   24     0.58     0.1008241   0.2316713   0.1188766          2        0.63        0.46        0.86

  CMV                  18     1.22     0.7950056   0.1485454   0.7088808          3        1.45        1.01        2.07
  any-reactivation                                                                                          
  ---------------------------------------------------------------------------------------------------------------------
